## <span style="color:#0b486b">Part 2: Deep Neural Networks (DNN) </span>
<div style="text-align: right"><span style="color:red; font-weight:bold">[Total marks for this part: 25 points]<span></div>

The second part of this assignment is to demonstrate your basis knowledge in deep learning that you have acquired from the lectures and tutorials materials. Most of the contents in this assignment are drawn from **the tutorials covered from weeks 1 to 2**. Going through these materials before attempting this assignment is highly recommended.

In the second part of this assignment, you are going to work with the FashionMNIST dataset for the image recognition task. It has the exact same format as MNIST (70,000 grayscale images of 28 × 28 pixels each with 10 classes), but the images represent fashion items rather than handwritten digits, so each class is more diverse, and the problem is significantly more challenging than MNIST.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(1234)

**Load the Fashion MNIST using `torchvision`**

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train_dataset_orgin = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

print(train_dataset_orgin.data.shape, train_dataset_orgin.targets.shape)
print(test_dataset.data.shape, test_dataset.targets.shape)

# Flatten the data
train_dataset_orgin.data = train_dataset_orgin.data.reshape(-1, 28*28)
test_dataset.data = test_dataset.data.reshape(-1, 28*28)

print(train_dataset_orgin.data.shape, train_dataset_orgin.targets.shape)
print(test_dataset.data.shape, test_dataset.targets.shape)

N = len(train_dataset_orgin)
print(f"Number of training samples: {N}")
N_train = int(0.9*N)
N_val = N - N_train
print(f"Number of training samples: {N_train}")
print(f"Number of validation samples: {N_val}")

train_dataset, val_dataset = torch.utils.data.random_split(train_dataset_orgin, [N_train, N_val])


print(len(train_dataset))
print(len(val_dataset))


train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

####  <span style="color:red">**Question 2.1:**</span> Write the code to visualize a mini-batch in `train_loader` including its images and labels.

<div style="text-align: right"> <span style="color:red">[5 points]</span> </div>

In [ ]:
import torch
from torchvision import datasets, transforms
import matplotlib.pyplot as plt


# Load dataset

transform = transforms.ToTensor()

dataset = datasets.FashionMNIST(
    root="./data",       # download path
    train=True,          # use training set
    download=True,       # download if not found
    transform=transform
)

train_loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)


# Visualization function

def visualize_mini_batch(train_loader, dataset, num_samples=8):
    dataiter = iter(train_loader)
    images, labels = next(dataiter)

    class_names = dataset.classes
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    axes = axes.ravel()

    for i in range(num_samples):
        img = images[i].squeeze().numpy()   # (1,28,28) → (28,28)
        label = labels[i].item()

        axes[i].imshow(img, cmap="gray")    # display as grayscale
        axes[i].set_title(class_names[label])
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()


# Call visualization

visualize_mini_batch(train_loader, dataset)


####<span style="color:red">**Question 2.2:**</span> Write the code for the feed-forward neural net using PyTorch

<div style="text-align: right"> <span style="color:red">[5 points]</span> </div>

We now develop a feed-forward neural network with the architecture $784 \rightarrow 40(ReLU) \rightarrow 30(ReLU) \rightarrow 10(softmax)$. You can choose your own way to implement your network and an optimizer of interest. You should train model in $50$ epochs and evaluate the trained model on the test set.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Dataset: MNIST

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)


# Define FeedForwardNet

class FeedForwardNet(nn.Module):
    def __init__(self, input_size=784, hidden1=40, hidden2=30, num_classes=10):
        super(FeedForwardNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten 28*28 -> 784
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)  # logits
        return x


# Initialize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FeedForwardNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop

num_epochs = 50
for epoch in range(num_epochs):
    # train
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * y.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    model.eval()
    test_loss, test_correct, test_total = 0, 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            test_loss += loss.item() * y.size(0)
            preds = logits.argmax(dim=1)
            test_correct += (preds == y).sum().item()
            test_total += y.size(0)

    test_loss /= test_total
    test_acc = test_correct / test_total

    print(f"Epoch {epoch+1}/{num_epochs} "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}% "
          f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc*100:.2f}%")


####  <span style="color:red">**Question 2.3:**</span> Tuning hyper-parameters with grid search
<div style="text-align: right"> <span style="color:red">[5 points]</span> </div>


Assume that you need to tune the number of neurons on the first and second hidden layers $n_1 \in \{20, 40\}$, $n_2 \in \{20, 40\}$  and the used activation function  $act \in \{sigmoid, tanh, relu\}$. The network has the architecture pattern $784 \rightarrow n_1 (act) \rightarrow n_2(act) \rightarrow 10(softmax)$ where $n_1, n_2$, and $act$ are in their grides. Write the code to tune the hyper-parameters $n_1, n_2$, and $act$. Note that you can freely choose the optimizer and learning rate of interest for this task.

In [ ]:
import torch.optim as optim
from itertools import product

class FlexibleNet(nn.Module):
    def __init__(self, input_size=784, n1=40, n2=30, num_classes=10, activation='relu'):
        super(FlexibleNet, self).__init__()
        self.fc1 = nn.Linear(input_size, n1)
        self.fc2 = nn.Linear(n1, n2)
        self.fc3 = nn.Linear(n2, num_classes)

        # Set activation function
        if activation == 'sigmoid':
            self.activation = torch.sigmoid
        elif activation == 'tanh':
            self.activation = torch.tanh
        else:  # relu
            self.activation = F.relu

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc3(x)
        return x

def train_and_evaluate(model, train_loader, val_loader, epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model.train()
    for epoch in range(epochs):
        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

    # Evaluate on validation set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in val_loader:
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

    accuracy = 100 * correct / total
    return accuracy

# Grid search
n1_options = [20, 40]
n2_options = [20, 40]
activation_options = ['sigmoid', 'tanh', 'relu']

best_accuracy = 0
best_params = {}

print("Grid Search Results:")
print("-" * 50)

for n1, n2, act in product(n1_options, n2_options, activation_options):
    model = FlexibleNet(n1=n1, n2=n2, activation=act)
    accuracy = train_and_evaluate(model, train_loader, val_loader)

    print(f"n1={n1}, n2={n2}, activation={act}: {accuracy:.2f}%")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_params = {'n1': n1, 'n2': n2, 'activation': act}

print(f"\nBest parameters: {best_params}")
print(f"Best accuracy: {best_accuracy:.2f}%")

####  <span style="color:red">**Question 2.4:**</span> Implement the loss with the form: $loss(p,y)=CE(1_{y},p)+\lambda H(p)$ where $H(p)=-\sum_{i=1}^{M}p_{i}\log p_{i}$ is the entropy of $p$, $p$ is the prediction probabilities of a data point $x$ with the ground-truth label $y$, $1_y$ is an one-hot label, and $\lambda >0$ is a trade-off parameter. Set $\lambda = 0.1$ to train a model.

<div style="text-align: right"> <span style="color:red">[5 points]</span> </div>


In [ ]:
class CustomLoss(nn.Module):
    def __init__(self, lambda_reg=0.1):
        super(CustomLoss, self).__init__()
        self.lambda_reg = lambda_reg
        self.cross_entropy = nn.CrossEntropyLoss()

    def forward(self, predictions, targets):
        # Cross-entropy loss: CE(1_y, p)
        ce_loss = self.cross_entropy(predictions, targets)

        # Entropy regularization: λH(p) = -λ Σ p_i log p_i
        # Apply softmax to get probabilities
        p = F.softmax(predictions, dim=1)

        # Calculate entropy H(p) = -Σ p_i log p_i
        # Add small epsilon to avoid log(0)
        epsilon = 1e-8
        entropy = -torch.sum(p * torch.log(p + epsilon), dim=1)
        entropy_reg = self.lambda_reg * torch.mean(entropy)

        # Total loss
        total_loss = ce_loss + entropy_reg

        return total_loss

# Usage example
model = FeedForwardNet()
custom_criterion = CustomLoss(lambda_reg=0.1)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop with custom loss
def train_with_custom_loss(model, train_loader, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data)
            loss = custom_criterion(output, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f'Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.6f}')

        avg_loss = running_loss / len(train_loader)
        print(f'Epoch {epoch+1} completed. Average Loss: {avg_loss:.6f}')

# Train model with custom loss
train_with_custom_loss(model, train_loader)

####  <span style="color:red">**Question 2.5:**</span> Experimenting with **sharpness-aware minimization** technique
<div style="text-align: right"> <span style="color:red">[5 points]</span> </div>

Sharpness-aware minimization (SAM) (i.e., [link for main paper](https://openreview.net/pdf?id=6Tm1mposlrM) from Google Deepmind) is a simple yet but efficient technique to improve the generalization ability of deep learning models on unseen data examples. In your research or your work, you might potentially use this idea. Your task is to read the paper and implement *Sharpness-aware minimization (SAM)*. Finally, you need to apply SAM to the best architecture found in **Question 2.3**.   

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


# Sharpness-Aware Minimization (SAM) Optimizer
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs):
        """
        SAM optimizer implementation.
        :param params: model parameters
        :param base_optimizer: the base optimizer (e.g., SGD, Adam)
        :param rho: neighborhood size
        :param adaptive: whether to use adaptive SAM
        """
        assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"

        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super(SAM, self).__init__(params, defaults)

        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        """Perform the ascent step (w + e(w))."""
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)

            for p in group["params"]:
                if p.grad is None:
                    continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (torch.pow(p, 2) if group["adaptive"] else 1.0) * p.grad * scale.to(p)
                p.add_(e_w)  # climb to the local maximum

        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        """Restore weights and update with the base optimizer."""
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                p.data = self.state[p]["old_p"]  

        self.base_optimizer.step()

        if zero_grad:
            self.zero_grad()

    def step(self, closure=None):
        """Performs one SAM optimization step."""
        assert closure is not None, "SAM requires closure (a full forward-backward pass)"

        # First forward-backward pass (retain computation graph)
        loss = closure(backward=True)

        # Perturb the parameters
        self.first_step(zero_grad=True)

        # Second forward-backward pass
        closure(backward=True)

        # Restore parameters and update
        self.second_step()

        return loss

    def _grad_norm(self):
        """Compute gradient norm."""
        shared_device = self.param_groups[0]["params"][0].device
        norm = torch.norm(
            torch.stack([
                ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad).norm(dtype=torch.float32).to(shared_device)
                for group in self.param_groups for p in group["params"]
                if p.grad is not None
            ]),
            dtype=torch.float32
        )
        return norm


# Training function with SAM
def train_with_sam(model, train_loader, epochs=5):
    """
    Train a model with Sharpness-Aware Minimization (SAM).
    :param model: the neural network
    :param train_loader: training data loader
    :param epochs: number of training epochs
    """
    criterion = nn.CrossEntropyLoss()
    base_optimizer = optim.SGD
    optimizer = SAM(model.parameters(), base_optimizer, lr=0.01, momentum=0.9)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):

            # Define closure for SAM
            def closure(backward=True):
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                if backward:
                    loss.backward(retain_graph=True)  # retain computation graph for second backward
                return loss

            loss = optimizer.step(closure)
            running_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f'Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.6f}')

        avg_loss = running_loss / len(train_loader)
        print(f'Epoch {epoch+1} completed. Average Loss: {avg_loss:.6f}')

model_sam = FeedForwardNet()
print("Training with Sharpness-Aware Minimization:")
train_with_sam(model_sam, train_loader)
